# 双向 GRU + Bahdanau 注意力的 Seq2Seq

这个 notebook 使用基础 PyTorch 完成英法机器翻译，不依赖 d2l 的训练封装。

核心结构：双向 Encoder GRU → Bahdanau 加性注意力 → Decoder GRU。

In [ ]:
import math
from collections import Counter

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils import data

torch.manual_seed(42)

num_epochs = 300
batch_size = 64
num_steps = 20
learning_rate = 0.005
num_examples = 600

data_path = "../data/fra-eng/fra.txt"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("device:", device)

## 1. 读取、清洗和分词

In [ ]:
def data_read(path: str) -> str:
    with open(path, "r", encoding="utf-8") as file:
        return file.read()


def preprocess_nmt(text: str) -> str:
    text = (
        text
        .replace("\u202f", " ")
        .replace("\xa0", " ")
        .lower()
    )

    result = []

    for i, char in enumerate(text):
        if (
            char in ",.!?"
            and i > 0
            and text[i - 1] != " "
        ):
            result.append(" ")

        result.append(char)

    return "".join(result)


def tokenize_nmt(text: str, max_text: int = 600):
    source = []
    target = []

    for line in text.splitlines():
        parts = line.split("\t")

        if len(parts) < 2:
            continue

        source.append(parts[0].split())
        target.append(parts[1].split())

        if max_text is not None and len(source) >= max_text:
            break

    return source, target

## 2. 词表、编号、截断与填充

In [ ]:
def vocab_maker(tokens, min_freq: int = 2):
    counter = Counter()

    for line in tokens:
        counter.update(line)

    token_freq = sorted(
        counter.items(),
        key=lambda item: (-item[1], item[0])
    )

    idx_to_token = [
        "<unk>",
        "<pad>",
        "<bos>",
        "<eos>",
    ]

    for token, freq in token_freq:
        if freq < min_freq:
            continue

        if token not in idx_to_token:
            idx_to_token.append(token)

    token_to_idx = {
        token: idx
        for idx, token in enumerate(idx_to_token)
    }

    return {
        "idx_to_token": idx_to_token,
        "token_to_idx": token_to_idx,
    }


def truncate_pad(line, step_num: int, pad_idx: int):
    if len(line) > step_num:
        return line[:step_num]

    return line + [pad_idx] * (step_num - len(line))


def build_array_nmt(lines, vocab, step_num: int):
    token_to_idx = vocab["token_to_idx"]

    unk_idx = token_to_idx["<unk>"]
    pad_idx = token_to_idx["<pad>"]
    eos_idx = token_to_idx["<eos>"]

    array = []

    for line in lines:
        line_idx = [
            token_to_idx.get(token, unk_idx)
            for token in line
        ]

        line_idx.append(eos_idx)
        line_idx = truncate_pad(
            line_idx,
            step_num,
            pad_idx,
        )

        array.append(line_idx)

    array = torch.tensor(array, dtype=torch.long)
    valid_len = (array != pad_idx).sum(dim=1)

    return array, valid_len


def load_data_nmt(
    batch_size: int,
    step_num: int,
    max_examples: int = 600,
):
    raw_text = data_read(data_path)
    text = preprocess_nmt(raw_text)
    source, target = tokenize_nmt(text, max_examples)

    src_vocab = vocab_maker(source, min_freq=2)
    tgt_vocab = vocab_maker(target, min_freq=2)

    src_array, src_valid_len = build_array_nmt(
        source,
        src_vocab,
        step_num,
    )

    tgt_array, tgt_valid_len = build_array_nmt(
        target,
        tgt_vocab,
        step_num,
    )

    dataset = data.TensorDataset(
        src_array,
        src_valid_len,
        tgt_array,
        tgt_valid_len,
    )

    train_iter = data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
    )

    return train_iter, src_vocab, tgt_vocab

In [ ]:
train_iter, src_vocab, tgt_vocab = load_data_nmt(
    batch_size,
    num_steps,
    num_examples,
)

src_pad_idx = src_vocab["token_to_idx"]["<pad>"]
tgt_pad_idx = tgt_vocab["token_to_idx"]["<pad>"]
tgt_bos_idx = tgt_vocab["token_to_idx"]["<bos>"]
tgt_eos_idx = tgt_vocab["token_to_idx"]["<eos>"]

X, X_valid_len, Y, Y_valid_len = next(iter(train_iter))

print("X shape:", X.shape)
print("X_valid_len shape:", X_valid_len.shape)
print("Y shape:", Y.shape)
print("Y_valid_len shape:", Y_valid_len.shape)
print("src vocab size:", len(src_vocab["idx_to_token"]))
print("tgt vocab size:", len(tgt_vocab["idx_to_token"]))

## 3. Masked Softmax 和 Bahdanau 加性注意力

Decoder 上一个隐藏状态是 Query；Encoder 每个时间步的输出同时作为 Key 和 Value。

In [ ]:
def masked_softmax(
    scores: torch.Tensor,
    valid_len: torch.Tensor,
) -> torch.Tensor:
    # scores:    (batch, query_num, key_num)
    # valid_len: (batch,)

    key_num = scores.shape[-1]

    positions = torch.arange(
        key_num,
        device=scores.device,
    ).reshape(1, 1, -1)

    mask = positions >= valid_len.reshape(-1, 1, 1)
    scores = scores.masked_fill(mask, -1e6)

    return torch.softmax(scores, dim=-1)


class AdditiveAttention(nn.Module):
    def __init__(
        self,
        query_size: int,
        key_size: int,
        hidden_size: int,
        dropout: float = 0.1,
    ):
        super().__init__()

        self.W_q = nn.Linear(
            query_size,
            hidden_size,
            bias=False,
        )

        self.W_k = nn.Linear(
            key_size,
            hidden_size,
            bias=False,
        )

        self.w_v = nn.Linear(
            hidden_size,
            1,
            bias=False,
        )

        self.dropout = nn.Dropout(dropout)
        self.attention_weights = None

    def forward(
        self,
        queries: torch.Tensor,
        keys: torch.Tensor,
        values: torch.Tensor,
        valid_len: torch.Tensor,
    ) -> torch.Tensor:
        queries = self.W_q(queries)
        keys = self.W_k(keys)

        # (batch, query_num, key_num, hidden_size)
        features = torch.tanh(
            queries.unsqueeze(2)
            + keys.unsqueeze(1)
        )

        # (batch, query_num, key_num)
        scores = self.w_v(features).squeeze(-1)

        self.attention_weights = masked_softmax(
            scores,
            valid_len,
        )

        # (batch, query_num, value_size)
        context = torch.bmm(
            self.dropout(self.attention_weights),
            values,
        )

        return context

## 4. Encoder、Decoder 和 Seq2Seq

In [ ]:
class Encoder(nn.Module):
    def __init__(
        self,
        input_num: int,
        hidden_size: int,
        embedding_size: int,
        pad_idx: int,
    ):
        super().__init__()

        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(
            input_num,
            embedding_size,
            padding_idx=pad_idx,
        )

        self.gru = nn.GRU(
            embedding_size,
            hidden_size,
            bidirectional=True,
        )

    def forward(self, x: torch.Tensor):
        # x: (batch, src_step)
        x = self.embedding(x)

        # (src_step, batch, embedding_size)
        x = x.permute(1, 0, 2)

        output, state = self.gru(x)

        # output: (src_step, batch, hidden_size*2)
        # state:  (2, batch, hidden_size)
        return output, state

In [ ]:
class Decoder(nn.Module):
    def __init__(
        self,
        output_num: int,
        hidden_size: int,
        embedding_size: int,
        pad_idx: int,
        encoder_output_size: int,
        attention_hidden_size: int,
    ):
        super().__init__()

        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(
            output_num,
            embedding_size,
            padding_idx=pad_idx,
        )

        self.attention = AdditiveAttention(
            query_size=hidden_size,
            key_size=encoder_output_size,
            hidden_size=attention_hidden_size,
            dropout=0.1,
        )

        self.gru = nn.GRU(
            embedding_size + encoder_output_size,
            hidden_size,
        )

        self.output_layer = nn.Linear(
            hidden_size,
            output_num,
        )

        self.attention_weights = []

    def forward(
        self,
        x: torch.Tensor,
        state: torch.Tensor,
        encoder_output: torch.Tensor,
        src_valid_len: torch.Tensor,
    ):
        # x: (batch, tgt_step)
        x = self.embedding(x)

        # (tgt_step, batch, embedding_size)
        x = x.permute(1, 0, 2)

        outputs = []
        self.attention_weights = []

        for token_embedding in x:
            # Decoder上一步隐藏状态作为Query
            # (batch, 1, hidden_size)
            query = state[-1].unsqueeze(1)

            # Encoder输出同时作为Key和Value
            # (batch, 1, encoder_output_size)
            context = self.attention(
                query,
                encoder_output,
                encoder_output,
                src_valid_len,
            )

            self.attention_weights.append(
                self.attention.attention_weights.detach()
            )

            # (batch, 1, embedding_size)
            token_embedding = token_embedding.unsqueeze(1)

            # 当前目标词Embedding与动态context合并
            # (batch, 1, embedding_size+encoder_output_size)
            gru_input = torch.cat(
                (token_embedding, context),
                dim=2,
            )

            # GRU要求时间步在第0维
            gru_input = gru_input.permute(1, 0, 2)
            output, state = self.gru(gru_input, state)
            outputs.append(output)

        # (tgt_step, batch, hidden_size)
        output = torch.cat(outputs, dim=0)

        # 时间步优先展开，与Y.T.reshape(-1)顺序一致
        output = output.reshape(-1, self.hidden_size)
        logits = self.output_layer(output)

        return logits, state

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(
        self,
        encoder: Encoder,
        decoder: Decoder,
        bos_idx: int,
    ):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder
        self.bos_idx = bos_idx

        # 双向Encoder的两个最终状态合并后，转换成Decoder初始状态
        self.bridge = nn.Linear(
            encoder.hidden_size * 2,
            decoder.hidden_size,
        )

    def make_decoder_state(
        self,
        encoder_state: torch.Tensor,
    ) -> torch.Tensor:
        forward_state = encoder_state[0]
        backward_state = encoder_state[1]

        state = torch.cat(
            (forward_state, backward_state),
            dim=1,
        )

        state = torch.tanh(self.bridge(state))

        return state.unsqueeze(0).repeat(
            self.decoder.gru.num_layers,
            1,
            1,
        )

    def forward(
        self,
        x: torch.Tensor,
        src_valid_len: torch.Tensor,
        y: torch.Tensor,
    ):
        encoder_output, encoder_state = self.encoder(x)
        decoder_state = self.make_decoder_state(encoder_state)

        # (src_step,batch,hidden*2) -> (batch,src_step,hidden*2)
        encoder_output = encoder_output.permute(1, 0, 2)

        # 教师强制：<bos> + Y[:, :-1]
        bos = torch.full(
            size=(y.shape[0], 1),
            fill_value=self.bos_idx,
            dtype=torch.long,
            device=y.device,
        )

        decoder_x = torch.cat(
            (bos, y[:, :-1]),
            dim=1,
        )

        logits, decoder_state = self.decoder(
            decoder_x,
            decoder_state,
            encoder_output,
            src_valid_len,
        )

        return logits, decoder_state

In [ ]:
encoder_hidden_size = 256
encoder_embedding_size = 128
decoder_hidden_size = 512
decoder_embedding_size = 256
attention_hidden_size = 128

my_encoder = Encoder(
    input_num=len(src_vocab["idx_to_token"]),
    hidden_size=encoder_hidden_size,
    embedding_size=encoder_embedding_size,
    pad_idx=src_pad_idx,
).to(device)

my_decoder = Decoder(
    output_num=len(tgt_vocab["idx_to_token"]),
    hidden_size=decoder_hidden_size,
    embedding_size=decoder_embedding_size,
    pad_idx=tgt_pad_idx,
    encoder_output_size=encoder_hidden_size * 2,
    attention_hidden_size=attention_hidden_size,
).to(device)

net = Seq2Seq(
    my_encoder,
    my_decoder,
    tgt_bos_idx,
).to(device)

loss_fn = nn.CrossEntropyLoss(
    ignore_index=tgt_pad_idx
)

optimizer = torch.optim.Adam(
    net.parameters(),
    lr=learning_rate,
)

print(net)

## 5. 训练

In [ ]:
def train_model(
    model: nn.Module,
    data_iter,
    epochs: int,
    optimizer: torch.optim.Optimizer,
    loss_fn: nn.Module,
    device: torch.device,
):
    history = []

    for epoch in range(epochs):
        model.train()

        train_loss = 0.0
        train_token_count = 0

        for x, x_valid_len, y, y_valid_len in data_iter:
            x = x.to(device)
            x_valid_len = x_valid_len.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            logits, _ = model(
                x,
                x_valid_len,
                y,
            )

            # Decoder输出按时间步优先展开，所以答案也这样展开
            target = y.T.reshape(-1)
            loss = loss_fn(logits, target)

            loss.backward()

            nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )

            optimizer.step()

            valid_token_num = y_valid_len.sum().item()
            train_loss += loss.item() * valid_token_num
            train_token_count += valid_token_num

        average_loss = train_loss / train_token_count
        history.append(average_loss)

        if epoch == 0 or (epoch + 1) % 10 == 0:
            print(
                f"epoch {epoch + 1}, "
                f"loss {average_loss:.4f}"
            )

    return history

In [ ]:
# 开始训练
loss_history = train_model(
    net,
    train_iter,
    num_epochs,
    optimizer,
    loss_fn,
    device,
)

plt.plot(loss_history)
plt.xlabel("epoch")
plt.ylabel("loss")
plt.show()

## 6. 贪心预测

从 `<bos>` 开始，每一步把当前预测作为下一步输入，预测到 `<eos>` 时停止。

In [ ]:
def predict_seq2seq(
    text: str,
    model: Seq2Seq,
    src_vocab,
    tgt_vocab,
    step_num: int,
    device: torch.device,
):
    model.eval()

    text = preprocess_nmt(text)
    tokens = text.split()

    src_token_to_idx = src_vocab["token_to_idx"]
    tgt_token_to_idx = tgt_vocab["token_to_idx"]

    src_unk_idx = src_token_to_idx["<unk>"]
    src_eos_idx = src_token_to_idx["<eos>"]
    src_pad_idx = src_token_to_idx["<pad>"]

    bos_idx = tgt_token_to_idx["<bos>"]
    eos_idx = tgt_token_to_idx["<eos>"]

    x = [
        src_token_to_idx.get(token, src_unk_idx)
        for token in tokens
    ]

    x.append(src_eos_idx)

    src_valid_len = torch.tensor(
        [min(len(x), step_num)],
        dtype=torch.long,
        device=device,
    )

    source_tokens = (tokens + ["<eos>"])[:step_num]
    x = truncate_pad(x, step_num, src_pad_idx)

    x = torch.tensor(
        x,
        dtype=torch.long,
        device=device,
    ).reshape(1, -1)

    output_idx = []
    attention_history = []

    with torch.no_grad():
        encoder_output, encoder_state = model.encoder(x)
        decoder_state = model.make_decoder_state(encoder_state)

        # (1, src_step, encoder_output_size)
        encoder_output = encoder_output.permute(1, 0, 2)

        decoder_x = torch.tensor(
            [[bos_idx]],
            dtype=torch.long,
            device=device,
        )

        for _ in range(step_num):
            logits, decoder_state = model.decoder(
                decoder_x,
                decoder_state,
                encoder_output,
                src_valid_len,
            )

            attention_history.append(
                model.decoder.attention.attention_weights[
                    0, 0
                ].cpu()
            )

            predict_idx = logits.argmax(dim=1).item()

            if predict_idx == eos_idx:
                break

            output_idx.append(predict_idx)

            decoder_x = torch.tensor(
                [[predict_idx]],
                dtype=torch.long,
                device=device,
            )

    result = [
        tgt_vocab["idx_to_token"][idx]
        for idx in output_idx
    ]

    if attention_history:
        attention_weights = torch.stack(attention_history)
    else:
        attention_weights = torch.empty((0, step_num))

    return " ".join(result), attention_weights, source_tokens

In [ ]:
# 翻译示例
test_sentences = [
    "go .",
    "i lost .",
    "he is calm .",
]

for sentence in test_sentences:
    translation, _, _ = predict_seq2seq(
        sentence,
        net,
        src_vocab,
        tgt_vocab,
        num_steps,
        device,
    )

    print(f"{sentence} => {translation}")

## 7. 查看注意力权重

横轴是英文输入位置，纵轴是 Decoder 的生成时间步。颜色越亮，注意力权重越大。

In [ ]:
def show_attention(text: str):
    translation, weights, source_tokens = predict_seq2seq(
        text,
        net,
        src_vocab,
        tgt_vocab,
        num_steps,
        device,
    )

    target_tokens = translation.split()

    if weights.shape[0] == len(target_tokens) + 1:
        target_tokens.append("<eos>")

    weights = weights[:, :len(source_tokens)]

    plt.figure(figsize=(8, 4))
    plt.imshow(weights.numpy(), cmap="Blues", aspect="auto")
    plt.colorbar()
    plt.xticks(range(len(source_tokens)), source_tokens)
    plt.yticks(range(len(target_tokens)), target_tokens)
    plt.xlabel("source tokens")
    plt.ylabel("generated tokens")
    plt.title(f"{text} -> {translation}")
    plt.show()


show_attention("go .")